# Parte 2 · Da matriz de respostas à IRT

**Avaliação com habilidade latente · Instituto Kunumi**

Uma população de 24 modelos faz a mesma prova de 90 exemplos, alguns com rótulo trocado de propósito. Ajustamos um modelo de IRT de dois parâmetros (2PL) e lemos o que ele diz sobre modelos e exemplos.

**Execute a célula de preparação primeiro.** Depois siga as tarefas em ordem. O código já está pronto: execute, observe e escreva a sua análise. Mude o que quiser para testar hipóteses; as funções de `utils/handson.py` estão documentadas (`help(funcao)` ou Shift + Tab no Jupyter e no Colab).

In [ ]:
import subprocess
import sys
from pathlib import Path

# No Colab, baixa o repositório do tutorial; localmente, encontra a raiz a partir desta pasta.
if "google.colab" in sys.modules:
    ROOT = Path("/content/latent-ability-ml")
    if not ROOT.exists():
        subprocess.check_call(["git", "clone", "-q", "https://github.com/manuelfjr/latent-ability-ml.git", str(ROOT)])
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "handson.py").exists())

sys.path.insert(0, str(ROOT))
print("Raiz do projeto:", ROOT)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "figure.facecolor": "#F0F0F0",
    "axes.facecolor": "#F0F0F0",
    "savefig.facecolor": "#F0F0F0",
    "axes.edgecolor": "#B4ADA4",
    "axes.labelcolor": "#1C2127",
    "axes.titlecolor": "#1C2127",
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#B4ADA4",
    "grid.alpha": 0.35,
    "xtick.color": "#5E5E5E",
    "ytick.color": "#5E5E5E",
    "text.color": "#1C2127",
    "legend.frameon": False,
    "lines.linewidth": 2.6,
    "axes.prop_cycle": mpl.cycler(color=["#F04E44", "#4392D2", "#2FB7CF", "#62CCA3", "#F0D449"]),
})

# Escala sequencial de um só tom (azul da paleta de dados) para mapas de dificuldade.
from matplotlib.colors import LinearSegmentedColormap

if "kunumi_azul" not in mpl.colormaps:
    mpl.colormaps.register(LinearSegmentedColormap.from_list("kunumi_azul", ["#B9D2EA", "#4392D2", "#14304F"]))

## Tarefa 1 · Leia três itens (~3 min)

Três curvas características do item (CCI) do 2PL, P(θ) = 1 / (1 + exp(−a(θ − δ))). A faixa cinza marca modelos com habilidade entre 0 e 1.

**Pergunta.** Qual item você usaria para distinguir um modelo com θ = 0 de outro com θ = 1? E qual usaria para separar modelos muito fracos? Justifique lendo as curvas e a tabela.

**Para explorar.** No simulador acima, crie um item que não serve para comparar ninguém. Que combinações de δ e a produzem isso?

In [ ]:
import numpy as np
import pandas as pd
from utils.handson import binary_irt_probability

theta = np.linspace(-3, 3, 200)   # habilidade latente
itens = {"A": (-1.0, 1.0), "B": (0.5, 1.0), "C": (0.5, 4.0)}   # nome: (dificuldade δ, discriminação a)

fig, ax = plt.subplots()
ax.axvspan(0, 1, color="#B4ADA4", alpha=0.25, label="modelos entre θ = 0 e θ = 1")
for nome, (d, a) in itens.items():
    ax.plot(theta, binary_irt_probability(theta, d, a), label=f"item {nome}: δ = {d}, a = {a}")
ax.axhline(0.5, color="#B4ADA4", linestyle="--", linewidth=1)
ax.set_xlabel("habilidade θ")
ax.set_ylabel("P(acerto)")
ax.set_title("Três itens, três comportamentos")
ax.legend(fontsize=8)
plt.show()

pd.DataFrame({nome: binary_irt_probability(np.array([-1.0, 0.0, 1.0]), d, a) for nome, (d, a) in itens.items()},
             index=["θ = −1", "θ = 0", "θ = 1"]).round(2)

*Sua análise:* 

## Tarefa 2 · Uma população de modelos (~3 min)

O mesmo tipo de problema, agora com 8% dos rótulos trocados de propósito (e registrados). Quatro famílias de classificadores, cada uma treinada com 3%, 6%, 12%, 25%, 50% ou 100% do conjunto de treino: 24 respondentes para a mesma prova.

**Pergunta.** Quais modelos você esperava no topo e na base? A acurácia separa bem os modelos do meio da tabela?

**Para explorar.** Passe o mouse sobre `make_model_population` e `flip_labels` e veja o que dá para mudar. O que você espera que aconteça com mais rótulos trocados?

In [ ]:
from utils.handson import (
    evaluate_models_on_dataset,
    flip_labels,
    make_model_population,
    make_toy_classification_dataset,
)

dados = make_toy_classification_dataset(n_samples=300, class_sep=1.4, flip_y=0.0, random_state=56)
dados = flip_labels(dados, fraction=0.08, random_state=0)   # 8% dos rótulos trocados, de propósito

populacao = make_model_population()   # 4 famílias × 6 frações do conjunto de treino = 24 modelos
res = evaluate_models_on_dataset(dados, scenario="ita", models=populacao, random_state=56)
R = res.pivot(index="model", columns="example_id", values="correct")
print(R.shape, "→ (modelos, exemplos de teste)")

acuracia = R.mean(axis=1).sort_values()
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(acuracia.index, acuracia.values, color="#4392D2", height=0.7)
ax.set_xlim(0.4, 1.0)
ax.set_xlabel("acurácia no teste")
ax.set_title("24 modelos, uma prova de 90 questões")
ax.tick_params(axis="y", labelsize=8)
plt.show()

*Sua análise:* 

## Tarefa 3 · Habilidade não é acurácia (~4 min)

Ajustamos o 2PL na matriz 24 × 90 com `fit_binary_2pl`, que estima uma habilidade θ por modelo e uma dificuldade δ e uma discriminação a por exemplo. A função `compara` mostra a dificuldade dos exemplos que só um de dois modelos acerta.

**Pergunta.** Habilidade e acurácia contam a mesma história? Encontre dois modelos com acurácia (quase) igual e θ diferente: o que explica a diferença?

**Para explorar.** Use `compara` com outros pares. Você consegue achar um par em que o modelo de menor acurácia tem θ maior?

In [ ]:
from utils.handson import fit_binary_2pl

habilidade, itens = fit_binary_2pl(R)   # θ por modelo; δ e a por exemplo de teste

comparacao = pd.DataFrame({"acurácia": R.mean(axis=1), "habilidade θ": habilidade}).sort_values("habilidade θ")
print("correlação de Spearman:", round(comparacao.corr(method="spearman").iloc[0, 1], 2))

fig, ax = plt.subplots()
ax.scatter(comparacao["acurácia"], comparacao["habilidade θ"], s=40, color="#4392D2", edgecolor="#1C2127", linewidth=0.5)
ax.set_xlabel("acurácia")
ax.set_ylabel("habilidade θ")
ax.set_title("Mesma prova, duas réguas")
plt.show()


def compara(a, b):
    """Dificuldade dos exemplos que só um dos dois modelos acerta."""
    so_a = R.columns[(R.loc[a] == 1) & (R.loc[b] == 0)]
    so_b = R.columns[(R.loc[a] == 0) & (R.loc[b] == 1)]
    print(f"só {a} acerta:", itens.loc[so_a, "difficulty"].round(2).tolist())
    print(f"só {b} acerta:", itens.loc[so_b, "difficulty"].round(2).tolist())


compara("logística · 100% dos dados", "árvore · 50% dos dados")   # <- troque os nomes
comparacao.round(2)

*Sua análise:* 

## Tarefa 4 · O que torna um exemplo difícil? (~4 min)

A dificuldade δ estimada de cada exemplo de teste, no plano das features. Os × marcam os rótulos que trocamos de propósito, uma informação que a IRT *não* viu.

**Pergunta.** Onde ficam os exemplos mais difíceis? Quantos dos mais difíceis são rótulos trocados, e o que explica os demais? Que uso prático você vê nisso?

**Para explorar.** Procure os × que *não* estão entre os mais difíceis. Olhe a discriminação deles na tarefa seguinte.

In [ ]:
teste = dados.set_index("example_id").loc[R.columns].assign(
    dificuldade=itens["difficulty"], discriminacao=itens["discrimination"]
)

fig, ax = plt.subplots()
pontos = ax.scatter(teste.feature_1, teste.feature_2, c=teste.dificuldade, cmap="kunumi_azul",
                    s=40, edgecolor="#5E5E5E", linewidth=0.3)
trocados = teste[teste.label_flipped]
ax.scatter(trocados.feature_1, trocados.feature_2, marker="x", s=110, linewidth=2.2, color="#F04E44",
           label="rótulo trocado")
fig.colorbar(pontos, ax=ax, label="dificuldade δ")
ax.set_title("Dificuldade de cada exemplo de teste")
ax.legend(fontsize=8)
plt.show()

teste.sort_values("dificuldade", ascending=False).head(8)[["dificuldade", "discriminacao", "label", "label_flipped"]].round(2)

*Sua análise:* 

## Tarefa 5 · Que exemplos realmente medem algo? (~3 min)

Exemplos que todos acertam ou todos erram não separam ninguém. Para os demais, vemos dificuldade e discriminação juntas.

**Pergunta.** Que fração da prova realmente informa? Se você só pudesse aplicar 15 exemplos para ranquear novos modelos, quais escolheria, e o que faria com os de discriminação negativa?

**Para explorar.** Os exemplos com discriminação negativa são acertados mais por modelos fracos do que por fortes. Quantos deles são rótulos trocados (tarefa anterior)?

In [ ]:
print("todos acertam:", int((R.min(axis=0) == 1).sum()))
print("todos erram:  ", int((R.max(axis=0) == 0).sum()))
informativos = R.columns[(R.min(axis=0) == 0) & (R.max(axis=0) == 1)]   # pelo menos um acerto e um erro
print("informativos: ", len(informativos), "de", R.shape[1])

fig, ax = plt.subplots()
ax.scatter(itens.loc[informativos, "difficulty"], itens.loc[informativos, "discrimination"],
           s=40, color="#4392D2", edgecolor="#1C2127", linewidth=0.5)
ax.axhline(0, color="#B4ADA4", linewidth=1)
ax.set_xlabel("dificuldade δ")
ax.set_ylabel("discriminação a")
ax.set_title("Os exemplos que separam os modelos")
plt.show()

itens.loc[informativos].sort_values("discrimination").head(10).round(2)

*Sua análise:* 

## Livre · Livre: sua análise (~5 min)

Todos os objetos das tarefas anteriores continuam disponíveis (`R`, `habilidade`, `itens`, `teste`, `fit_binary_2pl`). O código abaixo é só um ponto de partida: tira da prova os exemplos suspeitos e reajusta o modelo.

**Pergunta.** Que pergunta você investigou, como, e o que concluiu? O ranking dos modelos mudou ao limpar a prova?

**Para explorar.** Outras ideias: monte uma prova curta com os k melhores itens e compare o ranking com o da prova completa; aumente a fração de rótulos trocados e veja quantos a IRT ainda acha; troque o critério de suspeito.

In [ ]:
# Um ponto de partida: o ranking dos modelos muda se tirarmos da prova os exemplos suspeitos?
suspeitos = itens.index[(itens.discrimination < 0) | (itens.difficulty > itens.difficulty.quantile(0.95))]
habilidade_sem, _ = fit_binary_2pl(R.drop(columns=suspeitos))

print(len(suspeitos), "exemplos removidos; deles,", int(teste.loc[suspeitos, "label_flipped"].sum()), "tinham rótulo trocado")
pd.DataFrame({
    "posição antes": habilidade.rank(ascending=False).astype(int),
    "posição depois": habilidade_sem.rank(ascending=False).astype(int),
}).sort_values("posição depois")

*Sua análise:* 

## Para ir além: e quando não há rótulos?

Em agrupamento não existe acerto, então não há matriz de acertos. O **CLAIRE** troca acerto por *concordância*: para cada ponto i e algoritmo j, p_ij ∈ [0, 1] mede quanto j concorda com os demais sobre com quem i foi agrupado. Como essas respostas são contínuas, o β4-IRT estima a habilidade de cada algoritmo e a dificuldade de cada ponto, sem usar rótulo nenhum.

A demonstração está no fim da Parte 2 no site (código em `notebooks/ita/claire_demo.py`). Referência: Ferreira-Junior et al. (2025), *CLAIRE: clustering evaluation based on item response theory and model agreement*, Machine Learning 114, 256. https://doi.org/10.1007/s10994-025-06911-0